In [ ]:
"""
Calculates out vwap using duckdb directly
"""


import duckdb
from pathlib import Path
import gc

BASE = Path.cwd()

while BASE.name != "market-analytics-platform":
    BASE = BASE.parent

DATA_PATH = BASE / "data" / "**" / "*.snappy.parquet"

con = duckdb.connect()
con.execute("INSTALL delta; LOAD delta;")



In [ ]:
from datetime import datetime


df = con.execute(f"""
            SELECT 
                product_id,
                SUM(price * last_size) / SUM(last_size) as vwap
            FROM read_parquet('{DATA_PATH}', hive_partitioning=true)
            WHERE year = ? AND month = ? AND day = ?
            AND product_id = ?
            GROUP BY product_id
        """, [datetime.now().year, datetime.now().month, "01", "BTC-USD"]).fetchdf().dropna()
df

In [ ]:
df = None
gc.collect()